# Instance AI — Workflow-Builder Failure Analysis

Reads `data/classified/<window>.json` (produced by `classify.py`) and renders a dashboard of failure modes for the `workflow-builder` sub-agent.

**To re-run for a different window:** edit `WINDOW` below and re-execute.

In [ ]:
WINDOW = '20260421-20260428'  # set this to the window you want

# LangSmith URL parameters — used to deep-link traces. Override if your tenant differs.
LANGSMITH_TENANT_ID = 'a7b41b9d-bee0-4cf0-9786-f4600380f803'
LANGSMITH_PROJECT_ID = 'cbeb4b95-5d9c-40b4-81f5-6239a427d632'  # instance-ai
LANGSMITH_HOST = 'https://smith.langchain.com'

import json, sys
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown, HTML

DATA = Path('data/classified') / f'{WINDOW}.json'
blob = json.loads(DATA.read_text())
summary = blob['summary']
builders = pd.DataFrame(blob['builders'])
threads = pd.DataFrame(blob['threads'])

def trace_url(trace_id):
    return f'{LANGSMITH_HOST}/o/{LANGSMITH_TENANT_ID}/projects/p/{LANGSMITH_PROJECT_ID}/r/{trace_id}?poll=false'

def thread_url(thread_id):
    # Search runs by thread_id metadata; user lands in project filtered to the thread.
    return f'{LANGSMITH_HOST}/o/{LANGSMITH_TENANT_ID}/projects/p/{LANGSMITH_PROJECT_ID}?searchModel=%7B%22filter%22%3A%22and(eq(metadata_key%2C%5C%22thread_id%5C%22)%2Ceq(metadata_value%2C%5C%22{thread_id}%5C%22))%22%7D'

print(f'Window:    {summary["window"]}')
print(f'Builders:  {summary["totals"]["builders"]}')
print(f'Threads:   {summary["totals"]["threads"]}')
print(f'Failed:    {summary["totals"]["builders_failed"]}')
print(f'Cancelled: {summary["totals"]["builders_cancelled"]}')
print(f'\nThresholds (p95 ∨ floor):')
for k, v in summary['thresholds'].items():
    print(f'  {k:14s} = {v}')

## 1. Distributions

Where do builders sit on the curve? Where are the cliffs? Threshold lines are dashed.

In [ ]:
metrics = [
    ('llm_calls',     'Step count (LLM calls)',          summary['thresholds']['llm_calls']),
    ('latency_s',     'Latency (s)',                     summary['thresholds']['latency_s']),
    ('tokens',        'Total tokens',                    summary['thresholds']['tokens']),
    ('exec_calls',    'mastra_workspace_execute_command', summary['thresholds']['exec_calls']),
    ('edit_calls',    'mastra_workspace_edit_file',      summary['thresholds']['edit_calls']),
    ('write_calls',   'mastra_workspace_write_file',     summary['thresholds']['write_calls']),
    ('submit_calls',  'submit-workflow',                 summary['thresholds']['submit_calls']),
]

from plotly.subplots import make_subplots
fig = make_subplots(rows=4, cols=2, subplot_titles=[m[1] for m in metrics] + [''])
for i, (col, _label, thr) in enumerate(metrics):
    row, c = i // 2 + 1, i % 2 + 1
    vals = builders[col].dropna()
    fig.add_trace(go.Histogram(x=vals, nbinsx=40, marker_color='#5b8def', showlegend=False), row=row, col=c)
    fig.add_vline(x=thr, line_dash='dash', line_color='crimson', row=row, col=c)
fig.update_layout(height=900, title_text='Per-builder metric distributions (red dashed = threshold)', bargap=0.05)
fig.show()

## 2. Violation counts (per thread vs. per builder)

A single bad thread often contains multiple bad builders, so per-thread is the more honest "how many distinct user sessions had this problem" view.

In [ ]:
vc_b = summary['violation_counts_per_builder']
vc_t = summary['violation_counts_per_thread']
all_codes = sorted(set(vc_b) | set(vc_t))

df_v = pd.DataFrame({
    'code': all_codes,
    'per_builder': [vc_b.get(c, 0) for c in all_codes],
    'per_thread': [vc_t.get(c, 0) for c in all_codes],
})
df_v['pct_threads'] = (df_v['per_thread'] / summary['totals']['threads'] * 100).round(1)
df_v = df_v.sort_values('per_thread', ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(y=df_v['code'], x=df_v['per_thread'], orientation='h', name='threads', marker_color='#ef5b5b'))
fig.add_trace(go.Bar(y=df_v['code'], x=df_v['per_builder'], orientation='h', name='builders', marker_color='#5b8def', opacity=0.6))
fig.update_layout(barmode='overlay', height=420, title='Violations: distinct threads vs total builder events')
fig.show()

display(df_v.sort_values('per_thread', ascending=False).reset_index(drop=True))

## 3. Co-occurrence matrix

Which violations show up together? Cell `(i, j)` = number of threads that triggered both `i` and `j`. Diagonal = total threads with `i`. Strong off-diagonal blocks suggest correlated codes that we may want to collapse into one.

In [ ]:
all_codes_t = sorted(vc_t.keys())
co = pd.DataFrame(0, index=all_codes_t, columns=all_codes_t, dtype=int)
for _, row in threads.iterrows():
    codes = row['violation_codes'] or []
    for a in codes:
        for b in codes:
            if a in co.index and b in co.columns:
                co.loc[a, b] += 1

fig = px.imshow(
    co.values,
    x=co.columns, y=co.index,
    color_continuous_scale='Reds',
    text_auto=True,
    aspect='auto',
    title='Per-thread violation co-occurrence (cell = #threads with both)'
)
fig.update_layout(height=520)
fig.show()

# Jaccard similarity between codes — easier to spot correlated pairs.
import numpy as np
diag = np.diag(co.values).astype(float)
with np.errstate(divide='ignore', invalid='ignore'):
    union = diag[:, None] + diag[None, :] - co.values
    jac = np.where(union > 0, co.values / union, 0.0)
np.fill_diagonal(jac, 0.0)
fig2 = px.imshow(jac, x=co.columns, y=co.index, color_continuous_scale='Blues', text_auto='.2f', aspect='auto',
                title='Jaccard similarity (1.0 = always co-occur)')
fig2.update_layout(height=520)
fig2.show()

## 4. Top examples per failure mode

For each heuristic, the 10 worst builders by the underlying metric. Click the trace_id to open in LangSmith; the thread link filters the project to all runs in that conversation.

In [ ]:
PANEL_BY_CODE = {
    'step_explosion':  'llm_calls',
    'ts_error_loop':   'exec_calls',
    'edit_thrash':     'edit_calls',
    'write_thrash':    'write_calls',
    'submit_loop':     'submit_calls',
    'latency_outlier': 'latency_s',
    'token_outlier':   'tokens',
    'hard_failure':    None,
    'cancelled':       None,
}

builders['violation_codes'] = builders['violations'].apply(
    lambda vs: [v['code'] for v in (vs or [])]
)

for code, sort_col in PANEL_BY_CODE.items():
    sub = builders[builders['violation_codes'].apply(lambda xs: code in xs)].copy()
    if sub.empty:
        continue
    if sort_col:
        sub = sub.sort_values(sort_col, ascending=False)
    sub = sub.head(10)
    rows = []
    for _, b in sub.iterrows():
        rows.append({
            'when': (b['start_time'] or '')[:16].replace('T', ' '),
            'thread': f'<a href="{thread_url(b["thread_id"])}" target="_blank">{(b["thread_id"] or "?")[:8]}</a>' if b['thread_id'] else '?',
            'trace':  f'<a href="{trace_url(b["trace_id"])}" target="_blank">{b["trace_id"][:8]}</a>',
            'llm':    int(b['llm_calls']),
            'exec':   int(b['exec_calls']),
            'edit':   int(b['edit_calls']),
            'write':  int(b['write_calls']),
            'submit': int(b['submit_calls']),
            'latency_s': round(b['latency_s'], 0) if b['latency_s'] else None,
            'tokens': int(b['tokens'] or 0),
            'status': f'{b["status"]} / {b["final_status"]}',
        })
    df = pd.DataFrame(rows)
    display(Markdown(f'### `{code}` — top {len(df)} (sorted by `{sort_col or "recency"}`)'))
    display(HTML(df.to_html(escape=False, index=False)))


## 5. Per-thread drilldown

All threads sorted by violation count, then total tokens. Look here when a specific thread keeps showing up across multiple panels.

In [ ]:
drill = threads.copy()
drill['violation_count'] = drill['violation_codes'].apply(len)
drill['violations'] = drill['violation_codes'].apply(lambda xs: ', '.join(xs))
drill['link'] = drill['thread_id'].apply(lambda t: f'<a href="{thread_url(t)}" target="_blank">{t[:8]}</a>')
drill['first'] = drill['first_start'].str[:16].str.replace('T', ' ')
drill = drill[['link', 'first', 'builder_count', 'fail_count', 'total_tokens', 'violation_count', 'violations']]
drill = drill.rename(columns={'link': 'thread'})
drill = drill.sort_values(['violation_count', 'total_tokens'], ascending=[False, False])

display(Markdown(f'**{len(drill)} threads, top 30 shown**'))
display(HTML(drill.head(30).to_html(escape=False, index=False)))

## 6. Worst offenders by metric

Top builders for each individual metric, regardless of whether they crossed a threshold. Useful for spotting near-misses or extreme outliers.

In [ ]:
def top_by(col, n=10):
    sub = builders.sort_values(col, ascending=False).head(n).copy()
    sub['trace'] = sub['trace_id'].apply(lambda t: f'<a href="{trace_url(t)}" target="_blank">{t[:8]}</a>')
    sub['thread'] = sub['thread_id'].apply(lambda t: f'<a href="{thread_url(t)}" target="_blank">{(t or "?")[:8]}</a>' if t else '?')
    sub['violations'] = sub['violation_codes'].apply(lambda xs: ', '.join(xs))
    keep = ['thread', 'trace', col, 'llm_calls', 'exec_calls', 'edit_calls', 'write_calls', 'submit_calls', 'latency_s', 'tokens', 'status', 'final_status', 'violations']
    return sub[[c for c in keep if c in sub.columns]]

for col in ['llm_calls', 'latency_s', 'tokens', 'exec_calls', 'edit_calls', 'write_calls', 'submit_calls']:
    display(Markdown(f'### Top by `{col}`'))
    display(HTML(top_by(col).to_html(escape=False, index=False)))